# ⚔ Rise to Challenger — JupyterLab Launch Pad

Run cells **top to bottom**, once each. The app will be live after Step 5.

| Step | Action |
|------|--------|
| 1 | Install Python packages |
| 2 | Configure API keys |
| 3 | Check Ollama + models |
| 4 | Launch Streamlit |
| 5 | Get your access URL |

---
## Step 1 — Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -r requirements.txt -q
print('✅ All packages installed')

---
## Step 2 — Configure API Keys

**Replace the values below**, then run the cell. Your keys are saved to `.env` (never committed to git).

In [ ]:
# ── Fill in your keys here ────────────────────────────────────────────────────
RIOT_API_KEY      = "RGAPI-your-key-here"   # REQUIRED — https://developer.riotgames.com/
LANGCHAIN_API_KEY = ""                       # optional  — https://smith.langchain.com/
SERPER_API_KEY    = ""                       # optional  — https://serper.dev/
# ─────────────────────────────────────────────────────────────────────────────

env = f"""RIOT_API_KEY={RIOT_API_KEY}
LANGCHAIN_API_KEY={LANGCHAIN_API_KEY}
LANGCHAIN_PROJECT=rise-to-challenger
SERPER_API_KEY={SERPER_API_KEY}
"""
with open(".env", "w") as f:
    f.write(env)

if "your-key" in RIOT_API_KEY:
    print("⚠️  RIOT_API_KEY not set — replace it above, then re-run this cell")
else:
    print("✅ .env saved")

---
## Step 3 — Check Ollama

Ollama must be running on this server. If a model is missing, run `ollama pull <model>` in a terminal.

In [ ]:
import subprocess, time

REQUIRED_MODELS = ["gemma2:2b", "nomic-embed-text"]

# Check if Ollama is running; start it if not
result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=8)
if result.returncode != 0:
    print("Ollama not running — attempting to start...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=8)

if result.returncode == 0:
    lines = result.stdout.strip().split("\n")
    installed = [l.split()[0] for l in lines[1:] if l.strip()]
    print(f"✅ Ollama running — installed models: {installed}")
    missing = [m for m in REQUIRED_MODELS if not any(m in x for x in installed)]
    if missing:
        print(f"\n⚠️  Missing: {missing}")
        print("   Open a terminal and run:")
        for m in missing:
            print(f"     ollama pull {m}")
    else:
        print("   Required models present — AI coaching ready ✅")
else:
    print("❌ Cannot reach Ollama. Make sure it is installed on this server.")

---
## Step 4 — Launch Streamlit

Starts the app in the background. This cell returns immediately — the app keeps running.

In [ ]:
import subprocess, sys, time, os

PORT = 8501

# Kill any previous instance on this port
subprocess.run(f"lsof -ti:{PORT} | xargs kill -9 2>/dev/null; true", shell=True)
time.sleep(1)

log = open("streamlit.log", "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port",        str(PORT),
     "--server.address",     "0.0.0.0",
     "--server.headless",    "true",
     "--server.enableCORS",  "false"],
    stdout=log, stderr=subprocess.STDOUT,
    cwd=os.getcwd()
)

time.sleep(4)  # wait for startup

if proc.poll() is None:
    print(f"✅ Streamlit is running (PID {proc.pid}) on port {PORT}")
    print(f"   Logs saved to: streamlit.log")
else:
    print("❌ Streamlit crashed at startup. Log:")
    with open("streamlit.log") as f:
        print(f.read()[-3000:])

---
## Step 5 — Get Your Access URL

In [ ]:
import os, socket

PORT = 8501

# JupyterHub sets this env var — use it to build the proxy URL
hub_prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "")

print("═" * 60)
print("  RISE TO CHALLENGER — App Access")
print("═" * 60)

if hub_prefix:
    # JupyterHub environment — proxy URL is the correct one
    proxy = f"{hub_prefix}proxy/{PORT}/"
    server = os.environ.get("JUPYTERHUB_BASE_URL", "https://your-server.com")
    print(f"\n  ✅ JupyterHub proxy URL (share this):")
    print(f"     {server.rstrip('/')}{proxy}")
    print(f"\n  Or relative path only:")
    print(f"     {proxy}")
else:
    # Standalone JupyterLab / no Hub
    try:
        ip = socket.gethostbyname(socket.gethostname())
    except:
        ip = "<server-ip>"
    print(f"\n  Local:         http://localhost:{PORT}")
    print(f"  Server IP:     http://{ip}:{PORT}")
    print(f"\n  If on a remote server with a domain, try:")
    print(f"  https://<your-domain>/proxy/{PORT}/")
    print(f"  (requires jupyter-server-proxy to be installed on the server)")

print("\n" + "═" * 60)

---
## Utilities

In [ ]:
# View last 60 lines of Streamlit log (run this if something looks wrong)
with open("streamlit.log") as f:
    lines = f.readlines()
print("".join(lines[-60:]))

In [ ]:
# Stop Streamlit
import subprocess
subprocess.run("lsof -ti:8501 | xargs kill -9 2>/dev/null; true", shell=True)
print("⏹ Streamlit stopped")